In [1]:
!pip install selenium


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

All Sites

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

all_sites = []

driver.get("https://egymonuments.gov.eg/en/archaeological-sites")


# Waiting for the collection links to load
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))
i = 1
while True:
    try:
        # press the next site link
        try:
            site_link = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f'//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/a[{i}]')
            ))
            site_link.click()
            i += 1
        except:
            driver.find_element(By.XPATH, '//*[@id="cd-museums"]/app-egyptian-treasure/div/div[3]/div/button').click()
            continue

        # collect the data
        place_name = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div.mainPageTitle'))).text
        place_location = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.itemInfo'
        ).text
        place_description = driver.find_element(By.CSS_SELECTOR, 
            'div.relative.descriptionImageCont > div.DescriptionSection > div > div.txtSection'
        ).text

        try:
            img_elem = driver.find_element(By.CSS_SELECTOR,
                'body > div.innerMainBgCont > div > div:nth-child(1) > div > div.relative.descriptionImageCont > div.imageCont img'
            )
            place_photo = img_elem.get_attribute("src")
        except:
            place_photo = ""

        try:
            start_from = driver.find_element(By.CSS_SELECTOR,
                '#planVisit > div.mapHoursCont > div.openingHoursSec > div:nth-child(2) > div:nth-child(1) > p'
            ).text
            end_at = driver.find_element(By.CSS_SELECTOR,
                '#planVisit > div.mapHoursCont > div.openingHoursSec > div:nth-child(2) > div:nth-child(2) > p'
            ).text
        except:
            start_from, end_at = "Not Available", "Not Available"

        try:
            tickets_price = driver.find_element(By.CSS_SELECTOR, 'div.ticketPriceDetails > div').text
        except:
            tickets_price = "Free"

        all_sites.append({
            "place_name": place_name,
            "place_location": place_location,
            "place_description": place_description,
            "start_from": start_from,
            "end_at": end_at,
            "tickets_price": tickets_price,
            "photo_url": place_photo,
            "on_map": ""
        })

        # return to main
        driver.back()
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#cd-museums a")))


    except Exception as e:
        print("Finished — no more items.")
        break

print("all_sites:",len(all_sites))


Finished — no more items.
all_sites: 34


In [6]:
df = pd.DataFrame(all_sites)
df

,place_name,place_location,place_description,start_from,end_at,tickets_price,photo_url
0,kom El-dikka,Alexandria,In the heart of modern Alexandria lies the Kom...,09:00 AM,05:00 PM,FOREIGNERS:\nAdult: EGP 200 / Student: EGP 100...,https://egymonuments.gov.eg/media/8912/picture...
1,Qubbat Al-Hawa Cemetery,Aswan,Qubbat Al-Hawa Cemetery consists of several to...,07:00 AM,04:00 PM,FOREIGNER:\nAdult: EGP 200\nStudent: EGP 100\n...,https://egymonuments.gov.eg/media/8359/%D9%85%...
2,Elephantine,Aswan,Elephantine is an archaeological island on the...,08:00 AM,04:00 PM,FOREIGNER:\nAdult: EGP 200\nStudent: EGP 100\n...,https://egymonuments.gov.eg/media/8353/%D8%AC%...
3,Al Kab,Aswan,This is site of ancient city of Nekhbet. as th...,07:00 AM,05:00 PM,FOREIGNER:\nAdult: EGP 200\nStudent: EGP 100\n...,https://egymonuments.gov.eg/media/8350/%D8%A7%...
4,Aswan,Aswan,"Aswan, called Sunn by the ancient Egyptians, i...",Not Available,Not Available,Free,https://egymonuments.gov.eg/media/8343/%D8%A3%...
5,Al-Bahnasa Tombs,Al-Minya,Al-Bahansa is famous for its ancient cemeterie...,Not Available,Not Available,Free,https://egymonuments.gov.eg/media/8187/downloa...
6,Amada Temples,Aswan,"Today, the temple of Amada, the temple of al-D...",07:00 AM,04:00 PM,FOREIGNER:\nAdult: EGP 150\nStudent: EGP 75\n\...,https://egymonuments.gov.eg/media/7918/0288.jp...
7,Temples of Wadi al Sebua,Aswan,The site New Wadi al-Sebua results from the re...,07:00 AM,04:00 PM,FOREIGNER:\nAdult: EGP 150\nStudent: EGP 75\n\...,https://egymonuments.gov.eg/media/7905/dsc_017...
8,Abu Mina,Alexandria,The archaeological site of Abu Mina features a...,07:00 AM,07:00 PM,Free,https://egymonuments.gov.eg/media/7803/6.jpg?c...
9,Gebel al-Silsila,Aswan,Gebel al-Silsila is a mountainous region with ...,07:00 AM,04:00 PM,FOREIGNERS:\nAdult: EGP 100/ Student: EGP 50\n...,https://egymonuments.gov.eg/media/7778/img_202...


In [ ]:
from selenium.webdriver.support import expected_conditions as EC

for idx, row in df.iterrows():
    query = f"{row['place_name']}, {row['place_location']}, Egypt"
    print("Searching Maps for:", query)

    driver.get("https://www.google.com/maps/search/" + query.replace(" ", "+") + "?hl=en")

    try:
        WebDriverWait(driver, 15).until(
            lambda d: d.find_elements(By.CSS_SELECTOR, "h1.DUwDvf")
            or d.find_elements(By.CSS_SELECTOR, "div.Nv2PK")
        )
    except:
        pass

    if driver.find_elements(By.CSS_SELECTOR, "div.Nv2PK"):
        try:
            first_result = driver.find_element(By.CSS_SELECTOR, "div.Nv2PK a.hfpxzc")
            first_result.click()
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1.DUwDvf"))
            )
        except:
            pass

    time.sleep(1.5)

    df.at[idx, "on_map"] = driver.current_url

driver.quit()


In [7]:
df.to_csv('Ancient_Sites_En.csv', index=False)